# Stage 2 — Instruction Fine-Tuning (SFT)

This notebook continues from the Stage 1 LoRA adapter and trains on 132 policy-grounded instruction-response pairs. The dataset is formatted as conversational prompt-completion records, so loss is focused on the assistant completion.

In [ ]:
# Run on a Google Colab GPU runtime (T4 or better).
!pip -q install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
from pathlib import Path
import json, os, sys, torch

# Expected workflow: clone the GitHub repository, then open the notebook from the repo.
CANDIDATES = [Path.cwd(), Path('/content/domain-ai-assistant-finetuning')]
ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Repository root not found. Clone the repo into /content/domain-ai-assistant-finetuning or run from the repo root.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Repository:', ROOT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required for these notebooks.')

In [ ]:
from datasets import Dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from common import SYSTEM_PROMPT, generate_answer, read_jsonl, write_comparison_report

BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024
STAGE1_ADAPTER = ROOT / 'outputs/non_instruction_adapter'
SFT_ADAPTER = ROOT / 'outputs/sft_adapter'
if not STAGE1_ADAPTER.exists():
    raise FileNotFoundError('Run non_instruction_finetuning.ipynb first.')

## 1. Load and validate the instruction dataset

In [ ]:
rows = read_jsonl(ROOT / 'data/instruction_dataset.jsonl')
assert len(rows) >= 100
assert len({r['instruction'] for r in rows}) == len(rows)
print('Instruction examples:', len(rows))
print(rows[0])

## 2. Convert to conversational prompt-completion format

In [ ]:
formatted = []
for row in rows:
    formatted.append({
        'prompt': [
            {'role':'system', 'content':SYSTEM_PROMPT},
            {'role':'user', 'content':row['instruction']},
        ],
        'completion': [{'role':'assistant', 'content':row['response']}],
    })
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0])

## 3. Reload the base model and attach the Stage 1 adapter as trainable

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = PeftModel.from_pretrained(model, str(STAGE1_ADAPTER), is_trainable=True)
model.print_trainable_parameters()

## 4. Train the SFT adapter

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=SFTConfig(
        max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True,
        output_dir=str(ROOT / 'outputs/sft_training'),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=4,
        learning_rate=1e-4,
        warmup_ratio=0.05,
        logging_steps=2,
        eval_strategy='epoch',
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=42,
        report_to='none',
    ),
)
train_result = trainer.train()
print(train_result)
(ROOT / 'artifacts/sft_train_metrics.json').write_text(
    json.dumps(train_result.metrics, indent=2, default=str), encoding='utf-8'
)
trainer.save_state()

## 5. Save, evaluate, and generate the SFT comparison report

In [ ]:
SFT_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_ADAPTER)
tokenizer.save_pretrained(SFT_ADAPTER)
FastLanguageModel.for_inference(model)
questions = json.loads((ROOT / 'data/evaluation_questions.json').read_text())
sft_answers = [generate_answer(model, tokenizer, q) for q in questions]
(ROOT / 'artifacts/sft_outputs.json').write_text(json.dumps(sft_answers, indent=2), encoding='utf-8')
base_path = ROOT / 'artifacts/base_outputs.json'
base_answers = json.loads(base_path.read_text()) if base_path.exists() else ['Run Stage 1'] * len(questions)
write_comparison_report(
    ROOT / 'reports/sft_model_comparison.md',
    'Base Model vs Instruction Fine-Tuned Model',
    questions,
    [
        ('Base model answer', base_answers),
        ('Fine-tuned model answer', sft_answers),
        ('Which is better?', ['Complete after human review'] * len(questions)),
        ('Reason', ['Compare correctness, domain accuracy, clarity, safety, and helpfulness'] * len(questions)),
    ],
)
print('Saved adapter:', SFT_ADAPTER)
print('Updated reports/sft_model_comparison.md')

## Human review required

Open `reports/sft_model_comparison.md` and replace “Complete after review” with **SFT** or **Base**, plus a short reason. Focus on correctness, NovaDesk policy accuracy, clarity, safety, helpfulness, and whether the response is less generic.